In [2]:
import pandas as pd
import numpy as np

# ==============================
# 1. ĐỌC DỮ LIỆU GỐC
# ==============================

file_path = "D:/Chuyên đề 2/data/raw/xagusd_d.csv"

df = pd.read_csv(file_path)

print("5 dòng đầu:")
print(df.head())

print("\nKích thước dữ liệu:")
print(df.shape)

print("\nTên các cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu:")
print(df.dtypes)

5 dòng đầu:
         Date   Open   High    Low  Close
0  2010-01-04  16.88  17.55  16.82  17.42
1  2010-01-05  17.59  17.85  17.47  17.76
2  2010-01-06  17.75  18.21  17.73  18.20
3  2010-01-07  18.23  18.34  18.04  18.23
4  2010-01-08  18.23  18.49  18.02  18.48

Kích thước dữ liệu:
(4301, 5)

Tên các cột:
['Date', 'Open', 'High', 'Low', 'Close']

Kiểu dữ liệu:
Date      object
Open     float64
High     float64
Low      float64
Close    float64
dtype: object


In [5]:
# ==============================
# 2. CHUẨN HÓA NGÀY
# ==============================

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print(df.dtypes)
print("Ngày đầu:", df["Date"].min())
print("Ngày cuối:", df["Date"].max())

Date     datetime64[ns]
Open            float64
High            float64
Low             float64
Close           float64
dtype: object
Ngày đầu: 2010-01-04 00:00:00
Ngày cuối: 2026-08-31 00:00:00


In [6]:
# Sắp ngày cũ -> mới
df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
print(df.tail())

        Date   Open   High    Low  Close
0 2010-01-04  16.88  17.55  16.82  17.42
1 2010-01-05  17.59  17.85  17.47  17.76
2 2010-01-06  17.75  18.21  17.73  18.20
3 2010-01-07  18.23  18.34  18.04  18.23
4 2010-01-08  18.23  18.49  18.02  18.48
           Date     Open    High     Low   Close
4296 2026-08-25  68.9520  69.945  67.472  68.669
4297 2026-08-26  68.6470  69.733  67.530  68.127
4298 2026-08-27  68.2430  69.724  67.699  69.272
4299 2026-08-28  69.3320  71.113  66.121  66.454
4300 2026-08-31  65.7995  67.468  65.639  66.560


In [7]:
# ==============================
# 3. KIỂM TRA MISSING
# ==============================

print("\nSố missing của từng biến:")
print(df.isnull().sum())


Số missing của từng biến:
Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64


In [9]:
# ==============================
# 4. KIỂM TRA NGÀY BỊ TRÙNG
# ==============================

duplicates = df[df["Date"].duplicated(keep=False)]

print("\nSố ngày bị trùng:")
print(len(duplicates))

if len(duplicates) > 0:
    print(duplicates)


Số ngày bị trùng:
0


In [10]:
price_cols = ["Open", "High", "Low", "Close"]

invalid_price = df[
    (df[price_cols] <= 0).any(axis=1)
]

print("\nCác dòng có giá <= 0:")
print(invalid_price)


Các dòng có giá <= 0:
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []


In [11]:
# ==============================
# 5. KIỂM TRA LOGIC OHLC
# ==============================

invalid_ohlc = df[
    (df["High"] < df["Open"]) |
    (df["High"] < df["Close"]) |
    (df["High"] < df["Low"]) |
    (df["Low"] > df["Open"]) |
    (df["Low"] > df["Close"])
]

print("\nSố dòng lỗi OHLC:")
print(len(invalid_ohlc))

if len(invalid_ohlc) > 0:
    print(invalid_ohlc)


Số dòng lỗi OHLC:
0


In [12]:
# ==============================
# 6. KIỂM TRA KHOẢNG CÁCH NGÀY
# ==============================

df["day_gap"] = df["Date"].diff().dt.days

print(df["day_gap"].value_counts().sort_index())

day_gap
1.0    3417
2.0      14
3.0     838
4.0      31
Name: count, dtype: int64


In [13]:
df["price"] = df["Close"]

In [14]:
# ==============================
# 7. TẠO LOG PRICE
# ==============================

df["log_price"] = np.log(df["Close"])

In [17]:
# tạo Return và log Return
df["simple_return"] = df["Close"].pct_change() * 100
df["log_return"] =  np.log(df["Close"] / df["Close"].shift(1)) * 100


In [18]:
print(df.head())

        Date   Open   High    Low  Close  day_gap  price  log_price  \
0 2010-01-04  16.88  17.55  16.82  17.42      NaN  17.42   2.857619   
1 2010-01-05  17.59  17.85  17.47  17.76      1.0  17.76   2.876949   
2 2010-01-06  17.75  18.21  17.73  18.20      1.0  18.20   2.901422   
3 2010-01-07  18.23  18.34  18.04  18.23      1.0  18.23   2.903069   
4 2010-01-08  18.23  18.49  18.02  18.48      1.0  18.48   2.916689   

   simple_return  log_return  
0            NaN         NaN  
1       1.951780    1.932977  
2       2.477477    2.447286  
3       0.164835    0.164699  
4       1.371366    1.362048  


In [19]:
df["squared_return"] = df["log_return"] ** 2
df["abs_return"] = df["log_return"].abs()

In [20]:
returns = df["log_return"].dropna()

Q1 = returns.quantile(0.25)
Q3 = returns.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["outlier_flag"] = (
    (df["log_return"] < lower_bound) |
    (df["log_return"] > upper_bound)
)

print(
    df[df["outlier_flag"]][
        ["Date", "Close", "log_return"]
    ]
)

           Date   Close  log_return
12   2010-01-20  17.890   -4.748505
23   2010-02-04  15.270   -7.017096
31   2010-02-16  16.120    3.664339
74   2010-04-16  17.730   -3.654892
86   2010-05-04  17.870   -5.073354
...         ...     ...         ...
4271 2026-07-21  58.790    4.102401
4282 2026-08-05  62.054    4.154138
4291 2026-08-18  63.333   -3.806133
4292 2026-08-19  67.003    5.633087
4299 2026-08-28  66.454   -4.153081

[271 rows x 3 columns]


In [22]:
print("\nThống kê giá:")
print(df["Close"].describe())

print("\nThống kê log-return:")
print(df["log_return"].describe())


Thống kê giá:
count    4301.000000
mean       25.207168
std        12.888677
min        11.954500
25%        17.105000
50%        21.854000
75%        28.735000
max       116.849000
Name: Close, dtype: float64

Thống kê log-return:
count    4300.000000
mean        0.031174
std         2.008644
min       -32.894854
25%        -0.826810
50%         0.059550
75%         0.960929
max         8.790587
Name: log_return, dtype: float64


In [23]:
silver_price = df[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "log_price"
    ]
].copy()

In [24]:
print(silver_price.head())
print(silver_price.isnull().sum())

        Date   Open   High    Low  Close  log_price
0 2010-01-04  16.88  17.55  16.82  17.42   2.857619
1 2010-01-05  17.59  17.85  17.47  17.76   2.876949
2 2010-01-06  17.75  18.21  17.73  18.20   2.901422
3 2010-01-07  18.23  18.34  18.04  18.23   2.903069
4 2010-01-08  18.23  18.49  18.02  18.48   2.916689
Date         0
Open         0
High         0
Low          0
Close        0
log_price    0
dtype: int64


In [25]:
silver_return = df[
    [
        "Date",
        "Close",
        "log_price",
        "simple_return",
        "log_return",
        "squared_return",
        "abs_return"
    ]
].copy()

silver_return = silver_return.dropna(
    subset=["log_return"]
).reset_index(drop=True)

In [26]:
print(silver_return.head())

print(silver_return.isnull().sum())

        Date  Close  log_price  simple_return  log_return  squared_return  \
0 2010-01-05  17.76   2.876949       1.951780    1.932977        3.736399   
1 2010-01-06  18.20   2.901422       2.477477    2.447286        5.989207   
2 2010-01-07  18.23   2.903069       0.164835    0.164699        0.027126   
3 2010-01-08  18.48   2.916689       1.371366    1.362048        1.855174   
4 2010-01-11  18.55   2.920470       0.378788    0.378072        0.142939   

   abs_return  
0    1.932977  
1    2.447286  
2    0.164699  
3    1.362048  
4    0.378072  
Date              0
Close             0
log_price         0
simple_return     0
log_return        0
squared_return    0
abs_return        0
dtype: int64


In [29]:
# ==============================
# 8.  DATA ĐÃ ĐƯỢC XỬ LÝ
# ==============================

silver_price.to_csv("D:/Chuyên đề 2/data/processed/silver_price_clean.csv", index=False)

silver_return.to_csv("D:/Chuyên đề 2/data/processed/silver_return_clean.csv", index=False)

print("\n Lưu dữ liệu đã tiền xử lí.")


 Lưu dữ liệu đã tiền xử lí.
